# msb-eyeball-demo runbook

**Date recorded:** 2026-09-08  
**Status:** live — sandbox `herdr--eyeball` (local:203) still running

This runbook stands up and verifies an end-to-end demo of `herdr-plugin-msb`:
a microsandbox guest with a credential mount, an in-guest dev server,
a herdr workspace with a live guest shell pane, and an SSH port-forward
that reaches the guest from a remote laptop.

## Two-host topology

| Role | Hostname | Tailscale IP | OS/arch |
|---|---|---|---|
| Engine (sandbox host) | `engine-03` | `100.64.0.156` | Linux x86-64 |
| Minion / laptop | — | `100.64.0.35` | macOS arm64, FISH shell |

All commands below run on **engine-03** unless a cell header says **LAPTOP**.  
Engine working directory: `/home/newman/magic/herdr-plugin-msb`

## Preconditions and safety rules

1. **At most one live sandbox at a time.** Check before creating; check after teardown.
2. **Memory cap ≤ 2 GiB per sandbox.** Guest RAM is memfd-backed (resident, unswappable).
3. **Never run bare `go test ./...` or `go build ./...`.** Always use `make` targets.
   Bare test commands run one binary per package at full `GOMAXPROCS` on top of live VMs
   and have OOM-killed the entire login session — taking `dbus` and `ssh-agent` with it.
4. **Run `msb list` first and last** to confirm sandbox state before and after teardown.
5. **Commit with explicit paths only** (`git commit --only <paths>`). Never `git add -A`.
6. **Never run `git stash`, `git checkout --`, or `git reset --hard`** — restore with `cp` from a backup.

---
## Stage 0 — pre-flight: confirm nothing is already running

In [ ]:
msb list

NAME              IMAGE     STATUS     CREATED
herdr--eyeball    alpine    running    2026-09-08 11:32:03


If `msb list` shows **no rows**, you are starting fresh — proceed to Stage 1.  
If it shows `herdr--eyeball`, the demo sandbox is already up — skip ahead to whichever stage you need to verify.

---
## Stage 1 — worktree and herdr workspace

In [ ]:
# Create the linked worktree on branch demo/eyeball.
# UNVERIFIED — not re-run this session (worktree already exists at /home/newman/magic/herdr-plugin-msb-demo)
git worktree add /home/newman/magic/herdr-plugin-msb-demo -b demo/eyeball

In [ ]:
# Create the herdr workspace pointing at the demo worktree.
# UNVERIFIED — not re-run this session (workspace w8D already exists)
herdr worktree create \
  --cwd /home/newman/magic/herdr-plugin-msb \
  --branch demo/eyeball \
  --path /home/newman/magic/herdr-plugin-msb-demo \
  --label "herdr-plugin-msb demo" \
  --no-focus
# -> workspace w8D

**Note:** workspace w8 (the main herdr-plugin-msb workspace) was repointed off
the retired nexus3 tree with:

```sh
herdr worktree open --workspace w8 --path /home/newman/magic/herdr-plugin-msb --no-focus
```

That repoint fixed a stale pointer. If you restore the earliest session.json backup
(`session.json.backup.20260908_112826`) it will be undone. See the Teardown section.

In [ ]:
# Verify both worktrees are present.
git worktree list

~/magic/herdr-plugin-msb 232e5c4 [main]
~/magic/herdr-plugin-msb-demo 7b8c239 [demo/eyeball]


---
## Stage 2 — create the sandbox

In [ ]:
# Create the eyeball sandbox.
# UNVERIFIED — not re-run this session (sandbox already running as local:203)
./herdr-plugin-msb create \
  -name eyeball \
  -image alpine \
  -mem 2048 \
  -vcpus 2 \
  -port 45455 \
  -worktree /home/newman/magic/herdr-plugin-msb-demo
# Credential mount defaults to true; -cred flag omitted is equivalent to -cred true.

In [ ]:
# Confirm sandbox is running.
msb list

NAME              IMAGE     STATUS     CREATED
herdr--eyeball    alpine    running    2026-09-08 11:32:03


msb name: `herdr--eyeball` (prefix `herdr--` + logical name `eyeball`)  
ID: `local:203`  
Project: `herdr`

## Stage 3 — guest dev server

The guest runs a durable HTTP/1.0 server on port 45455. It is started inside the guest with:

```sh
nc -lk -p 45455 -e /tmp/handler-cat.sh >/dev/null 2>&1 & sleep 1
```

where `/tmp/handler-cat.sh` contains `cat /tmp/http-response.bin` and `/tmp/http-response.bin`
is a pre-written binary blob:
`HTTP/1.0 200 OK\r\nContent-Type: text/plain\r\nContent-Length: 25\r\n\r\neyeball-1788867133-22958\n`.

`nc -lk` forks a fresh handler process per connection without closing the listen socket, so
there is no gap between requests. `cat` issues a single `read`+`write` over the binary blob;
busybox `printf` is not used here because its multiple `write()` calls do not relay correctly
through busybox nc v1.37.0's `-e` handler relay. libkrun forwards guest port 45455 to the
host automatically.

**Engine-side curl (tests only the libkrun published port — not the laptop SSH forward):**

In [ ]:
curl -sS http://127.0.0.1:45455/ ; echo "exit:$?"

**Expected:** token `eyeball-1788867133-22958`, exit **0**.

The durable server sends `Content-Length: 25` and closes the connection cleanly after the
response body. curl exits 0 immediately — no `--max-time` or timeout handling needed.

Actual output from this session:

```
eyeball-1788867133-22958
exit:0
```

---
## Stage 4 — herdr space and guest pane

In [ ]:
# Create the herdr space (workspace w8E labelled "msb:eyeball").
# UNVERIFIED — not re-run this session (space already exists as w8E)
./herdr-plugin-msb space-create -project herdr eyeball
# -> workspace w8E  label: msb:eyeball

In [ ]:
# Confirm the guest pane is running the exec command.
herdr pane process-info --pane w8E:p3

{"id":"cli:pane:process_info","result":{"process_info":{"foreground_process_group_id":698150,"foreground_processes":[{"argv":["/home/newman/.local/bin/herdr-plugin-msb","exec","-pty","-project","herdr","eyeball","--","/bin/sh"],"cmdline":"/home/newman/.local/bin/herdr-plugin-msb exec -pty -project herdr eyeball -- /bin/sh","cwd":"/home/newman/magic/herdr-plugin-msb","name":"herdr-plugin-ms","pid":698150}],"pane_id":"w8E:p3","shell_pid":698150},"type":"pane_process_info"}}


The `argv` field shows `herdr-plugin-msb exec -pty -project herdr eyeball -- /bin/sh` — confirming
the pane is attached to the running sandbox. The TUI for workspace w8E has:

- **Tab 1, pane 1** — host shell (`newman@engine-03`)
- **Tab 1, pane 2** — Microsandbox notification pane
- **Tab 2, pane 3** (label: "Microsandbox guest shell") — guest shell inside `herdr--eyeball`

In workspace w8D both panes have `foreground_cwd=/home/newman/magic/herdr-plugin-msb-demo`
(the demo worktree on branch `demo/eyeball`).

In [ ]:
# In the guest pane (w8E tab 2), run to confirm you are inside the VM:
# cat /proc/1/comm
#
# Expected: init.krun
#
# On the host /proc/1/comm is "systemd". Inside the libkrun VM PID 1 is init.krun.
# This is the definitive in-guest indicator.
#
# Also run:
# hostname
#
# Expected: herdr--eyeball
#
# NOTE: these commands must be typed in the TUI guest pane — they cannot be run
# here because this shell is a host shell, not the guest.

---
## Stage 5 — port forward

### 5a. Engine side — declare (run on engine-03)

In [ ]:
./herdr-plugin-msb declare -port 45455 -host engine-03 -notify ; echo "exit:$?"

[{"id":3,"host":"engine-03","remote_port":45455,"local_port":45455,"remote_bind":"127.0.0.1","origin":"discovery","plugin_version":"0.1.0","created_unix_ms":1788871159626}]
exit:0

### 5b. Laptop side — local-agent (run on minion 100.64.0.35)

The laptop runs the local-agent daemon. It polls the engine's JSON state file via SSH
ControlMaster and opens a local `LISTEN` socket for each declared forward.

```fish
# LAPTOP — FISH shell, macOS arm64
# Binary built natively at /Users/newman/.local/bin/herdr-plugin-msb
herdr-plugin-msb local-agent \
  -target newman@100.64.0.156 \
  -control-path /tmp/herdr-agent-engine-03.ctl \
  -host engine-03 \
  -poll 5s
```

SSH ControlMaster PID on laptop: **92315** (at time of demo)  
ControlPath: `/tmp/herdr-agent-engine-03.ctl`

The agent opens `127.0.0.1:45455` on the laptop and forwards it to `127.0.0.1:45455` on engine-03.

---
## Stage 6 — verification

### 6a. Plugin verbs (engine-03)

In [ ]:
./herdr-plugin-msb list

[{"id":3,"host":"engine-03","remote_port":45455,"local_port":45455,"remote_bind":"127.0.0.1","origin":"discovery","plugin_version":"0.1.0","created_unix_ms":1788871159626}]

In [ ]:
./herdr-plugin-msb status

pending=1 acked=2

### 6b. Positive control — laptop curl reaches guest token

**The laptop fetch is the only valid test of the SSH port forward.**
An engine-side `curl http://127.0.0.1:45455/` hits the libkrun published port directly and
never crosses the tunnel — passing engine-side fetches say nothing about whether the laptop
forward is alive.

Run from **engine-03** (SSH to laptop, run curl there):

```sh
ssh 100.64.0.35 'sh -c "curl -sS --max-time 5 http://127.0.0.1:45455/"'
```

Expected stdout: `eyeball-1788867133-22958`  
Expected exit code: **0**

Actual output from this session (10 consecutive laptop hits, all exit 0):

```
eyeball-1788867133-22958
eyeball-1788867133-22958
eyeball-1788867133-22958
eyeball-1788867133-22958
eyeball-1788867133-22958
eyeball-1788867133-22958
```

### 6c. Negative control — forward cancelled, curl fails

Cancel the specific forward (leave master alive):

```sh
# LAPTOP
ssh -S /tmp/herdr-agent-engine-03.ctl -O cancel -L 45455:127.0.0.1:45455 newman@100.64.0.156
```

The same laptop curl then gives:

```
curl: (7) Failed to connect to 127.0.0.1 port 45455 after 0 ms: Connection refused
```

Re-establish with:

```sh
# LAPTOP
ssh -S /tmp/herdr-agent-engine-03.ctl -O forward -L 45455:127.0.0.1:45455 newman@100.64.0.156
```

Laptop curl succeeds again: `eyeball-1788867133-22958`, exit 0.

All three states were observed live in this session.

**Two-outcome proof:**
- Forward alive → curl body = `eyeball-1788867133-22958`, exit **0**
- Forward cancelled → curl exit **7** (Connection refused), no body

### 6d. Visual confirmation (operator step)

Open the herdr TUI:

```sh
herdr
```

- Navigate to workspace **w8E** (`msb:eyeball`) → Tab 2 → guest pane shows `/ #` prompt.
- Navigate to workspace **w8D** (`herdr-plugin-msb demo`) → both panes cwd is
  `/home/newman/magic/herdr-plugin-msb-demo` (branch `demo/eyeball`).

This step is not scriptable — it is an eyeball confirmation in the TUI. The operator does it.

---
## TEARDOWN

> **Do these steps in order.** Skipping the sandbox removal leaks VM RAM (memfd-backed,
> unswappable). Run `msb list` before and after.

### Step 1 — cancel the port forward and kill the SSH master (LAPTOP)

In [ ]:
# LAPTOP — run on minion (100.64.0.35)
# UNVERIFIED — not run this session (forward is still live; operator runs this)

# Cancel only the port forward (leaves master alive for other uses):
ssh -S /tmp/herdr-agent-engine-03.ctl -O cancel -L 45455:127.0.0.1:45455 newman@100.64.0.156

# Then exit the master entirely:
ssh -S /tmp/herdr-agent-engine-03.ctl -O exit dummy

### Step 2 — remove the sandbox (engine-03)

In [ ]:
# UNVERIFIED — not run this session (sandbox still running; operator runs this)
./herdr-plugin-msb rm -project herdr eyeball

In [ ]:
# UNVERIFIED — not run this session (sandbox still running at teardown time)
# Confirm sandbox is gone.
msb list
# Expected: empty (no rows)

### Step 3 — close herdr workspaces (engine-03)

In [ ]:
# UNVERIFIED — not run this session (workspaces still open; operator runs these)

# Close the guest shell workspace:
herdr workspace close w8E

# Close the demo worktree workspace:
herdr workspace close w8D

### Step 4 — remove the linked worktree (engine-03)

In [ ]:
# UNVERIFIED — not run this session (worktree still present; operator runs this)
git -C /home/newman/magic/herdr-plugin-msb worktree remove /home/newman/magic/herdr-plugin-msb-demo

### Step 5 — restore session.json (engine-03)

Three backups on disk:

```
~/.config/herdr/session.json                                   # live (9.9K)
~/.config/herdr/session.json.backup.20260908_112826            # 5.0K — earliest, pre-workspace-additions
~/.config/herdr/session.json.bak-space-verbs-20260908-120028   # 6.4K — post-space-verbs, pre-demo
```

> **WARNING — read before choosing a backup:**
> Workspace **w8** was repointed from the retired `nexus3` tree to
> `/home/newman/magic/herdr-plugin-msb` during this session
> (`herdr worktree open --workspace w8 --path /home/newman/magic/herdr-plugin-msb --no-focus`).
> That repoint fixed a stale pointer and should be kept.
>
> - `session.json.backup.20260908_112826` predates the repoint — restoring it **undoes w8's repoint**.
> - `session.json.bak-space-verbs-20260908-120028` was taken after the repoint — safer to use.

In [ ]:
# UNVERIFIED — not run this session; operator chooses which backup to restore.

# RECOMMENDED — keeps the w8 repoint:
cp ~/.config/herdr/session.json.bak-space-verbs-20260908-120028 ~/.config/herdr/session.json

# ALTERNATIVE — pre-workspace additions (UNDOES the w8 repoint, reverts to stale nexus3 pointer):
# cp ~/.config/herdr/session.json.backup.20260908_112826 ~/.config/herdr/session.json

# Restart herdr after restoring.

## Known caveats

### 1. Minion go.mod directive divergence

The repo's `go.mod` `go` directive was lowered to `1.24.13` locally on the minion
because the minion has Go 1.24.13 while the repo formally requires 1.25.0.
This is a local workaround and is not committed to main. The engine builds against 1.25.0.

### 2. `placement="tab"` schema status unverified

The `space-create` verb passes `placement="tab"` when opening the guest shell pane.
This flag has not been verified against a herdr 0.8.0 schema document — it works
in the live test but the schema acceptance is not confirmed.

### 3. TUI eyeball is an operator step

Visual confirmation that the guest pane shows `/ #` and the worktree panes show the
correct `cwd` cannot be captured programmatically. It requires opening the herdr TUI
and navigating to workspaces w8E and w8D. That is the operator's own final step.

### 4. busybox nc -lk -e relay quirk

busybox nc v1.37.0's `-e` flag relays handler output through an internal buffer that
only forwards the first `write()` block per connection when the handler uses busybox
`printf` (which issues one `write()` per format token). The workaround is to pre-write
the full HTTP response as a binary file and use `cat` (single `read`+`write`) as the
handler. Do not replace `cat /tmp/http-response.bin` with inline `printf` in handler-cat.sh.